# Create TSC Alliance Awards

**TSC Alliance (Tuberous Sclerosis Alliance)** (funder_id `4320306781`, IAMHRF
expanded, priority `321`; dup-id F4320319495 flagged in the registry memo; UK
Tuberous Sclerosis Association F4320320028 is a different org). tsalliance.org
funded-grant accordion panels via Firefox-headless (CF blocks chromium+requests).
122 awards FY2002-FY2025, 100% PI/title/institution/year/scheme; no per-record
amounts at source (§6.7; their 13 Crossref grant DOIs carry amounts separately).

Parquet: `s3://openalex-ingest/awards/tsc_alliance/tsc_alliance_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tsc_alliance_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/tsc_alliance/tsc_alliance_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.tsc_alliance_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.tsc_alliance_raw;

## Step 2: Create TSC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tsc_alliance_awards
USING delta
AS
WITH
tsc_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306781  -- Tuberous Sclerosis Alliance
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'tsc_alliance' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.tsc_alliance_raw g
    CROSS JOIN tsc_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'tsc_alliance' AND priority = 321;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    321 as priority
FROM openalex.awards.tsc_alliance_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(display_name) has_title, COUNT(lead_investigator) has_pi, COUNT(start_year) has_year
FROM openalex.awards.tsc_alliance_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'tsc_alliance' AND priority = 321;